# 14.11 同一個區塊多跑幾次，省下的是什麼？


想把一份訂單材料處理三次，最直接的做法是一個接一個排三個 Transformer 區塊，這種排法叫**堆疊**（stacking）。每個區塊就是[4.5](https://birdhackor.github.io/tiny-perceptron-vlm/4.5.html)的block，有注意力與FFN；三個區塊各有自己的參數，模型可以在不同深度學不同工作。

另一個安排是只保留一份區塊參數，把前一次輸出的狀態再交回它。這叫 **Looped Transformer（循環式 Transformer）**，也稱具有 recurrent depth（循環深度）的模型。三輪共用的是**同一組區塊參數**，每次處理的狀態可以不同，所以不是把完全相同的輸入重算三遍。

先用明確標為假設的數字算儲存與工作量。假設字的embedding表與輸出層共20萬參數，一個區塊100萬參數；先忽略其他開銷，每跑一個區塊記一單位工作。

| 設計 | 區塊參數份數 | 總參數 | 處理三次的區塊工作 |
| --- | ---: | ---: | ---: |
| 三個不共享區塊（堆疊） | 3 | 320萬 | 3單位 |
| 一個共享區塊跑三輪（循環） | 1 | 120萬 | 3單位 |

下圖左邊的區塊A、B、C各存一份參數；右邊共享區塊F的迴線帶回更新後的狀態，原輸入e每輪都再送進F（圖中的「每輪再讀」）。右邊跑滿三輪後，輸出層才讀s₃。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/loop-shared-depth.svg")))

下面用4.5的 `Block` 實際建兩種安排，數參數，也數區塊執行幾次。這裡先不另外送入原輸入e，[14.12](https://birdhackor.github.io/tiny-perceptron-vlm/14.12.html)再加。


In [ ]:
import torch
from torch import nn

from tiny_perceptron.model import Block, ModelConfig

torch.manual_seed(0)
config = ModelConfig(width=8)
stacked = nn.ModuleList([Block(config) for _ in range(3)])
shared = Block(config)
looped = nn.ModuleList([shared, shared, shared])
for name, blocks in [("堆疊", stacked), ("循環", looped)]:
    state = torch.randn(1, 3, 8)
    for block in blocks:
        state, _, _ = block(state)
    count = sum(p.numel() for p in blocks.parameters())
    print(name, "參數", count, "區塊執行", len(blocks), "次", "輸出形狀", list(state.shape))

`stacked` 用[4.6](https://birdhackor.github.io/tiny-perceptron-vlm/4.6.html)的清單寫法造出三個彼此獨立的block；`looped` 把同一個 `shared` 放進清單三次。內層迴圈讓狀態依序經過清單裡的每一個block，`state, _, _ =` 只留新的表示（4.5）。`p.numel()` 是一張參數表裡有幾個數字，`sum(p.numel() for p in blocks.parameters())` 把每張表的個數加起來，寫法和[1.1](https://birdhackor.github.io/tiny-perceptron-vlm/1.1.html)的清單寫法相同，只是直接交給 `sum`；`parameters()` 遇到同一張表只列一次，所以循環版放了三次 `shared`，也只算一份。輸出是堆疊2,520個參數、循環840個，正好三比一；兩邊都執行三次區塊，輸出形狀都是 `[1, 3, 8]`。

省下的是參數的儲存，以及最佳化器替每個參數另存的狀態（Adam 的 m、v，[5.9](https://birdhackor.github.io/tiny-perceptron-vlm/5.9.html)），不是把三輪計算變成一輪。訓練時的反向傳播要用到前向計算途中算出的中間特徵（activation，5.9提過）；若保留每一輪的中間特徵，訓練記憶體仍會隨輪數增加。相同參數量下可多做幾輪，但多做工作需要時間，也未必改善答案。

這與[14.6](https://birdhackor.github.io/tiny-perceptron-vlm/14.6.html)的權重共享有共同想法：讓同一份參數用在不止一處。14.6讓字的embedding表與輸出層共用一張表；這裡是同一個區塊在不同輪次共用一組參數，沿深度反覆更新狀態。它也不等於第15章的MoE（Mixture of Experts，專家混合，[15.4](https://birdhackor.github.io/tiny-perceptron-vlm/15.4.html)）：MoE把FFN換成好幾份專家FFN，每個token只交給其中幾份計算，選的是「誰來算」；循環選的是同一區塊反覆幾次。兩種設計可以組合。

為什麼值得研究？當部署受參數記憶體限制、但不同題目可容許不同計算時，固定一份參數再調輪數，提供另一種安排資源的方式。Huginn 論文展示了大型循環深度語言模型；論文中的效果屬於它的資料、架構與訓練設定。這個可能性不構成所有小模型、每一道題或更長輪數的保證。

練習把共享區塊改跑五輪：程式裡 `looped` 的清單寫五次 `shared`。先預測參數仍是840、區塊執行變成5次；換成表中的假設數字，參數仍是120萬，區塊工作變五單位。若三輪答案已最好，五輪便可能只增加成本；後面會用評估曲線而不是名稱決定輪數。

延伸閱讀：[Scaling up Test-Time Compute with Latent Reasoning，v2，2025](https://arxiv.org/abs/2502.05171v2)。
